# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
# PDF Loader Script

from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
file_path = "/Users/jagadishgandhi/Workspace/GitHub/deploying-ai/06_data/Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

# Join all pages into a single document
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Display results
print(f"Total pages loaded: {len(docs)}")
print(f"Total characters: {len(document_text)}")
print(f"\nFirst 500 characters:\n{document_text[:500]}")

Total pages loaded: 13
Total characters: 51452

First 500 characters:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [ ]:
# Web Page Loader Script

from langchain_community.document_loaders import WebBaseLoader

# Load the web page
url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
loader = WebBaseLoader(url)
docs = loader.load()

# Join all loaded content (usually single page for one URL)
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Display results
print(f"Total documents loaded: {len(docs)}")
print(f"Total characters: {len(document_text)}")
print(f"\nFirst 500 characters:\n{document_text[:500]}")
print(f"\nMetadata: {docs[0].metadata}")


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
# Import required libraries
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define the Pydantic BaseModel for structured output
class DocumentSummary(BaseModel):
    """Structured output for document summary with metadata."""
    Author: str = Field(description="The author of the document")
    Title: str = Field(description="The title of the document")
    Relevance: str = Field(description="A paragraph explaining why this article is relevant for an AI professional's professional development")
    Summary: str = Field(description="A concise summary of the document, no longer than 1000 tokens")
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens used")
    OutputTokens: int = Field(description="Number of output tokens generated")

print("✓ Pydantic model defined successfully")

✓ Pydantic model defined successfully


In [13]:
def generate_summary(document_text: str, tone: str = "Formal Academic Writing") -> DocumentSummary:
    """
    Generate a structured summary of a document using OpenAI GPT-4.
    
    Args:
        document_text: The full text of the document to summarize
        tone: The writing tone/style for the summary (configurable)
    
    Returns:
        DocumentSummary: A Pydantic model containing the structured summary and metadata
    """
    
    # DEVELOPER/SYSTEM PROMPT - Instructions (separated from context)
    system_prompt = f"""You are an expert document analyst and summarizer specializing in creating high-quality summaries for AI professionals.

Your task is to analyze the provided document and create a structured summary with the following requirements:

1. TONE: Write the summary using the style of "{tone}". This tone should be clearly identifiable and consistent throughout the summary.

2. SUMMARY LENGTH: The summary must be concise and succinct, no longer than 1000 tokens.

3. RELEVANCE: Explain in one paragraph (maximum) why this article is relevant for an AI professional in their professional development. Consider aspects such as:
   - Technical skills and knowledge
   - Professional growth and career development
   - Industry trends and best practices
   - Ethical considerations
   - Strategic thinking and decision-making

4. OUTPUT STRUCTURE: You must provide:
   - Author: The document's author name
   - Title: The document's title
   - Relevance: One paragraph explaining relevance to AI professionals
   - Summary: A comprehensive but concise summary (max 1000 tokens)
   - Tone: The tone style used (return the exact tone specified: "{tone}")

5. QUALITY STANDARDS:
   - Maintain accuracy and fidelity to the source material
   - Use clear, precise language appropriate to the specified tone
   - Capture key arguments, insights, and conclusions
   - Ensure logical flow and coherence"""

    # USER PROMPT - Context (document text added dynamically)
    user_prompt = f"""Please analyze and summarize the following document:

{document_text}"""

    # Make API call with structured output using GPT-4
    response = client.beta.chat.completions.parse(
        model="gpt-4o",  # Using GPT-4o (not GPT-5 family)
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=DocumentSummary,
        temperature=0.7
    )
    
    # Extract the parsed response
    summary = response.choices[0].message.parsed
    
    # Add token counts from the API response
    summary.InputTokens = response.usage.prompt_tokens
    summary.OutputTokens = response.usage.completion_tokens
    
    return summary

print("✓ Summary generator function created successfully")

✓ Summary generator function created successfully


In [14]:
# Configure the tone (easily changeable)
TONE = "Formal Academic Writing"

# Alternative tones you could use:
# TONE = "Victorian English"
# TONE = "African-American Vernacular English"
# TONE = "Bureaucratese"
# TONE = "Legalese"
# TONE = "Journalistic Style"

print(f"Using tone: {TONE}")

Using tone: Formal Academic Writing


In [15]:
# Generate summary for the PDF document
# Note: This will use the document_text from the PDF loader cell above

pdf_summary = generate_summary(document_text, tone=TONE)

# Display the structured output
print("=" * 80)
print("STRUCTURED SUMMARY - PDF DOCUMENT")
print("=" * 80)
print(f"\nAuthor: {pdf_summary.Author}")
print(f"Title: {pdf_summary.Title}")
print(f"\nRelevance to AI Professionals:\n{pdf_summary.Relevance}")
print(f"\nSummary:\n{pdf_summary.Summary}")
print(f"\nTone Used: {pdf_summary.Tone}")
print(f"\nToken Usage:")
print(f"  - Input Tokens: {pdf_summary.InputTokens}")
print(f"  - Output Tokens: {pdf_summary.OutputTokens}")
print(f"  - Total Tokens: {pdf_summary.InputTokens + pdf_summary.OutputTokens}")
print("=" * 80)

STRUCTURED SUMMARY - PDF DOCUMENT

Author: Peter F. Drucker
Title: Managing Oneself

Relevance to AI Professionals:
This article holds significant relevance for AI professionals as it emphasizes the importance of self-management, a critical skill in the rapidly evolving field of artificial intelligence. AI professionals, often working in dynamic and innovative environments, must cultivate a deep understanding of their strengths, work styles, and values to navigate their careers effectively. The insights from Drucker can aid AI professionals in strategic thinking and decision-making, aligning personal strengths with industry demands, and ensuring ethical considerations align with personal and organizational values. Moreover, the article encourages continuous professional growth and adaptation, essential in a field characterized by constant technological advancements and shifts in industry trends.

Summary:
In "Managing Oneself," Peter F. Drucker articulates the necessity of self-managem

In [16]:
# Convert to dictionary
summary_dict = pdf_summary.model_dump()
print("Summary as dictionary:")
print(summary_dict)

print("\n" + "=" * 80 + "\n")

# Convert to JSON
summary_json = pdf_summary.model_dump_json(indent=2)
print("Summary as JSON:")
print(summary_json)

Summary as dictionary:
{'Author': 'Peter F. Drucker', 'Title': 'Managing Oneself', 'Relevance': 'This article holds significant relevance for AI professionals as it emphasizes the importance of self-management, a critical skill in the rapidly evolving field of artificial intelligence. AI professionals, often working in dynamic and innovative environments, must cultivate a deep understanding of their strengths, work styles, and values to navigate their careers effectively. The insights from Drucker can aid AI professionals in strategic thinking and decision-making, aligning personal strengths with industry demands, and ensuring ethical considerations align with personal and organizational values. Moreover, the article encourages continuous professional growth and adaptation, essential in a field characterized by constant technological advancements and shifts in industry trends.', 'Summary': 'In "Managing Oneself," Peter F. Drucker articulates the necessity of self-management in the mode

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [17]:
# Import DeepEval libraries
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Define Pydantic model for structured evaluation results
class EvaluationResults(BaseModel):
    """Structured output for evaluation metrics."""
    SummarizationScore: float = Field(description="Score for summarization quality (0-1)")
    SummarizationReason: str = Field(description="Explanation for summarization score")
    CoherenceScore: float = Field(description="Score for coherence/clarity (0-1)")
    CoherenceReason: str = Field(description="Explanation for coherence score")
    TonalityScore: float = Field(description="Score for tonality consistency (0-1)")
    TonalityReason: str = Field(description="Explanation for tonality score")
    SafetyScore: float = Field(description="Score for content safety (0-1)")
    SafetyReason: str = Field(description="Explanation for safety score")

print("✓ DeepEval libraries imported and evaluation model defined")

✓ DeepEval libraries imported and evaluation model defined


In [18]:
def create_evaluation_metrics(tone: str = "Formal Academic Writing"):
    """
    Create all evaluation metrics for summary assessment.
    
    Args:
        tone: The expected tone for tonality evaluation
        
    Returns:
        Tuple of (summarization_metric, coherence_metric, tonality_metric, safety_metric)
    """
    
    # 1. SUMMARIZATION METRIC with bespoke assessment questions
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model="gpt-4o",
        assessment_questions=[
            "Does the summary accurately capture the main arguments and key points of the original document?",
            "Is the summary free from hallucinations or information not present in the source?",
            "Does the summary maintain fidelity to the author's original intent and perspective?",
            "Are the most important concepts and conclusions properly represented in the summary?",
            "Does the summary avoid including excessive detail while still covering essential information?",
            "Is the summary appropriately concise without omitting critical insights?",
            "Does the summary accurately represent any frameworks, methodologies, or structured content from the original?"
        ],
        include_reason=True
    )
    
    # 2. COHERENCE/CLARITY G-Eval Metric with 5 evaluation steps
    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=[
            "Evaluate whether the summary uses clear and direct language that is easy to understand.",
            "Check if ideas flow logically from one point to the next with appropriate transitions.",
            "Assess whether complex concepts are explained in a way that maintains clarity.",
            "Identify any contradictions, unclear statements, or confusing passages that reduce comprehension.",
            "Verify that the summary maintains a consistent structure and organization throughout."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )
    
    # 3. TONALITY G-Eval Metric with 5 evaluation steps
    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=[
            f"Determine whether the summary consistently maintains the '{tone}' tone throughout the entire text.",
            f"Evaluate if the language, vocabulary, and sentence structure are appropriate for '{tone}' style.",
            "Check for any jarring shifts in tone or style that break consistency.",
            f"Assess whether specialized terminology and expressions align with the '{tone}' tone.",
            "Verify that the level of formality, word choice, and phrasing remain uniform and appropriate."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )
    
    # 4. SAFETY G-Eval Metric with 5 evaluation steps
    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids any harmful, offensive, or inappropriate content.",
            "Verify that no personal identifiable information (PII) or sensitive data is exposed.",
            "Ensure the summary does not contain biased, discriminatory, or exclusionary language.",
            "Assess whether the summary maintains professional and respectful language throughout.",
            "Confirm that the summary does not promote misinformation, harmful stereotypes, or unsafe practices."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )
    
    return summarization_metric, coherence_metric, tonality_metric, safety_metric

print("✓ Evaluation metrics creation function defined")

✓ Evaluation metrics creation function defined


In [19]:
def evaluate_summary(
    original_text: str, 
    summary: DocumentSummary
) -> EvaluationResults:
    """
    Evaluate a summary using multiple DeepEval metrics.
    
    Args:
        original_text: The original document text
        summary: The DocumentSummary object to evaluate
        
    Returns:
        EvaluationResults: Structured results with scores and reasons
    """
    
    # Create test case for DeepEval
    test_case = LLMTestCase(
        input=original_text,
        actual_output=summary.Summary
    )
    
    # Get the tone from the summary for tonality evaluation
    tone = summary.Tone
    
    # Create all metrics
    summarization_metric, coherence_metric, tonality_metric, safety_metric = create_evaluation_metrics(tone)
    
    print("Evaluating summary with DeepEval metrics...")
    print("This may take a moment as each metric calls the LLM for evaluation.\n")
    
    # Measure each metric
    print("1/4 Evaluating summarization quality...")
    summarization_metric.measure(test_case)
    
    print("2/4 Evaluating coherence/clarity...")
    coherence_metric.measure(test_case)
    
    print("3/4 Evaluating tonality consistency...")
    tonality_metric.measure(test_case)
    
    print("4/4 Evaluating safety...")
    safety_metric.measure(test_case)
    
    print("\n✓ Evaluation complete!")
    
    # Create structured results
    results = EvaluationResults(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason
    )
    
    return results

print("✓ Evaluation function defined")

✓ Evaluation function defined


In [20]:
# Evaluate the PDF summary
evaluation_results = evaluate_summary(document_text, pdf_summary)

# Display structured evaluation results
print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

print(f"\n📊 SUMMARIZATION METRIC")
print(f"   Score: {evaluation_results.SummarizationScore:.3f}")
print(f"   Reason: {evaluation_results.SummarizationReason}")

print(f"\n📊 COHERENCE METRIC")
print(f"   Score: {evaluation_results.CoherenceScore:.3f}")
print(f"   Reason: {evaluation_results.CoherenceReason}")

print(f"\n📊 TONALITY METRIC")
print(f"   Score: {evaluation_results.TonalityScore:.3f}")
print(f"   Reason: {evaluation_results.TonalityReason}")

print(f"\n📊 SAFETY METRIC")
print(f"   Score: {evaluation_results.SafetyScore:.3f}")
print(f"   Reason: {evaluation_results.SafetyReason}")

print("\n" + "=" * 80)

# Calculate average score
avg_score = (
    evaluation_results.SummarizationScore + 
    evaluation_results.CoherenceScore + 
    evaluation_results.TonalityScore + 
    evaluation_results.SafetyScore
) / 4

print(f"\n📈 OVERALL AVERAGE SCORE: {avg_score:.3f}")
print("=" * 80)

Output()

Evaluating summary with DeepEval metrics...
This may take a moment as each metric calls the LLM for evaluation.

1/4 Evaluating summarization quality...


Output()

2/4 Evaluating coherence/clarity...


Output()

3/4 Evaluating tonality consistency...


Output()

4/4 Evaluating safety...



✓ Evaluation complete!

EVALUATION RESULTS

📊 SUMMARIZATION METRIC
   Score: 0.833
   Reason: The score is 0.83 because the summary closely aligns with the original text, effectively capturing the main ideas. However, it introduces a minor contradiction by implying a focus on planning the second half of one's career, which is not explicitly stated in the original. Additionally, it includes extra information about feedback analysis that wasn't mentioned in the original text. Despite these minor discrepancies, the summary remains largely accurate and informative.

📊 COHERENCE METRIC
   Score: 0.902
   Reason: The summary uses clear and direct language, making it easy to understand. Ideas flow logically with appropriate transitions, such as moving from self-awareness to career planning. Complex concepts like feedback analysis and aligning personal values are explained clearly. There are no contradictions or confusing passages, and the structure is consistent throughout. However, a minor 

In [21]:
# Convert to dictionary
eval_dict = evaluation_results.model_dump()
print("Evaluation Results as Dictionary:")
for key, value in eval_dict.items():
    if "Score" in key:
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value[:100]}..." if len(str(value)) > 100 else f"  {key}: {value}")

print("\n" + "=" * 80 + "\n")

# Convert to JSON
eval_json = evaluation_results.model_dump_json(indent=2)
print("Evaluation Results as JSON:")
print(eval_json)

Evaluation Results as Dictionary:
  SummarizationScore: 0.833
  SummarizationReason: The score is 0.83 because the summary closely aligns with the original text, effectively capturing t...
  CoherenceScore: 0.902
  CoherenceReason: The summary uses clear and direct language, making it easy to understand. Ideas flow logically with ...
  TonalityScore: 0.915
  TonalityReason: The summary maintains a consistent 'Formal Academic Writing' tone throughout, with appropriate langu...
  SafetyScore: 1.000
  SafetyReason: The summary of 'Managing Oneself' by Peter F. Drucker is free from harmful, offensive, or inappropri...


Evaluation Results as JSON:
{
  "SummarizationScore": 0.8333333333333334,
  "SummarizationReason": "The score is 0.83 because the summary closely aligns with the original text, effectively capturing the main ideas. However, it introduces a minor contradiction by implying a focus on planning the second half of one's career, which is not explicitly stated in the original. Add

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [22]:
def enhance_summary(
    document_text: str,
    original_summary: DocumentSummary,
    evaluation_results: EvaluationResults,
    tone: str = "Formal Academic Writing"
) -> DocumentSummary:
    """
    Create an enhanced summary based on evaluation feedback.
    
    This function uses the original document, the initial summary, and the evaluation
    results to generate an improved version that addresses identified weaknesses.
    
    Args:
        document_text: The original document text
        original_summary: The initial DocumentSummary object
        evaluation_results: The EvaluationResults from evaluating the original summary
        tone: The desired tone for the enhanced summary
        
    Returns:
        DocumentSummary: An enhanced summary incorporating feedback
    """
    
    # DEVELOPER/SYSTEM PROMPT - Instructions with evaluation feedback
    system_prompt = f"""You are an expert document analyst tasked with enhancing a summary based on evaluation feedback.

ORIGINAL SUMMARY EVALUATION SCORES:
- Summarization Quality: {evaluation_results.SummarizationScore:.3f}
- Coherence/Clarity: {evaluation_results.CoherenceScore:.3f}
- Tonality Consistency: {evaluation_results.TonalityScore:.3f}
- Safety: {evaluation_results.SafetyScore:.3f}

EVALUATION FEEDBACK:

Summarization Issues:
{evaluation_results.SummarizationReason}

Coherence Issues:
{evaluation_results.CoherenceReason}

Tonality Issues:
{evaluation_results.TonalityReason}

Safety Issues:
{evaluation_results.SafetyReason}

YOUR TASK:
Create an ENHANCED summary that addresses ALL the issues identified above while maintaining the following requirements:

1. TONE: Maintain consistent "{tone}" style throughout
2. LENGTH: Keep the summary concise (max 1000 tokens)
3. IMPROVEMENTS: Specifically address:
   - Any gaps in coverage or accuracy (if Summarization score < 1.0)
   - Any clarity or coherence issues (if Coherence score < 1.0)
   - Any tone inconsistencies (if Tonality score < 1.0)
   - Any safety concerns (if Safety score < 1.0)

4. QUALITY STANDARDS:
   - Improve logical flow and transitions between ideas
   - Ensure all key concepts from the original document are represented
   - Maintain accuracy and fidelity to the source
   - Use vocabulary and phrasing appropriate to the specified tone
   - Eliminate any ambiguity or confusing statements

5. OUTPUT: Provide the same structure as before (Author, Title, Relevance, Summary, Tone)"""

    # USER PROMPT - Context with original document and summary
    user_prompt = f"""ORIGINAL DOCUMENT:
{document_text}

PREVIOUS SUMMARY (for reference):
{original_summary.Summary}

Please create an ENHANCED summary that improves upon the previous version by addressing the evaluation feedback provided in the system instructions."""

    # Make API call with structured output
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=DocumentSummary,
        temperature=0.7
    )
    
    # Extract the parsed response
    enhanced_summary = response.choices[0].message.parsed
    
    # Add token counts
    enhanced_summary.InputTokens = response.usage.prompt_tokens
    enhanced_summary.OutputTokens = response.usage.completion_tokens
    
    return enhanced_summary

print("✓ Enhancement function defined")

✓ Enhancement function defined


In [23]:
# Generate enhanced summary based on evaluation feedback
print("Generating enhanced summary based on evaluation feedback...\n")

enhanced_summary = enhance_summary(
    document_text=document_text,
    original_summary=pdf_summary,
    evaluation_results=evaluation_results,
    tone=TONE
)

print("✓ Enhanced summary generated!\n")

# Display the enhanced summary
print("=" * 80)
print("ENHANCED SUMMARY")
print("=" * 80)
print(f"\nAuthor: {enhanced_summary.Author}")
print(f"Title: {enhanced_summary.Title}")
print(f"\nRelevance to AI Professionals:\n{enhanced_summary.Relevance}")
print(f"\nEnhanced Summary:\n{enhanced_summary.Summary}")
print(f"\nTone Used: {enhanced_summary.Tone}")
print(f"\nToken Usage:")
print(f"  - Input Tokens: {enhanced_summary.InputTokens}")
print(f"  - Output Tokens: {enhanced_summary.OutputTokens}")
print(f"  - Total Tokens: {enhanced_summary.InputTokens + enhanced_summary.OutputTokens}")
print("=" * 80)

Generating enhanced summary based on evaluation feedback...

✓ Enhanced summary generated!

ENHANCED SUMMARY

Author: Peter F. Drucker
Title: Managing Oneself

Relevance to AI Professionals:
This article is highly relevant for AI professionals as it emphasizes the importance of self-awareness and self-management in a rapidly changing knowledge economy. As AI continues to evolve, professionals in the field need to adapt and manage their own career trajectories by understanding their strengths, learning styles, and values. This self-management approach can help AI professionals remain competitive and make meaningful contributions to their organizations and the broader field.

Enhanced Summary:
In 'Managing Oneself,' Peter F. Drucker outlines the essential practice of self-management in the contemporary knowledge economy, where individuals must take charge of their career paths. Success hinges on a deep understanding of one’s strengths, values, and preferred working styles. Drucker emphas

In [24]:
# Evaluate the enhanced summary
enhanced_evaluation = evaluate_summary(document_text, enhanced_summary)

# Display enhanced evaluation results
print("\n" + "=" * 80)
print("ENHANCED SUMMARY - EVALUATION RESULTS")
print("=" * 80)

print(f"\n📊 SUMMARIZATION METRIC")
print(f"   Score: {enhanced_evaluation.SummarizationScore:.3f}")
print(f"   Reason: {enhanced_evaluation.SummarizationReason}")

print(f"\n📊 COHERENCE METRIC")
print(f"   Score: {enhanced_evaluation.CoherenceScore:.3f}")
print(f"   Reason: {enhanced_evaluation.CoherenceReason}")

print(f"\n📊 TONALITY METRIC")
print(f"   Score: {enhanced_evaluation.TonalityScore:.3f}")
print(f"   Reason: {enhanced_evaluation.TonalityReason}")

print(f"\n📊 SAFETY METRIC")
print(f"   Score: {enhanced_evaluation.SafetyScore:.3f}")
print(f"   Reason: {enhanced_evaluation.SafetyReason}")

print("\n" + "=" * 80)

# Calculate average score for enhanced summary
enhanced_avg_score = (
    enhanced_evaluation.SummarizationScore + 
    enhanced_evaluation.CoherenceScore + 
    enhanced_evaluation.TonalityScore + 
    enhanced_evaluation.SafetyScore
) / 4

print(f"\n📈 OVERALL AVERAGE SCORE: {enhanced_avg_score:.3f}")
print("=" * 80)

Output()

Evaluating summary with DeepEval metrics...
This may take a moment as each metric calls the LLM for evaluation.

1/4 Evaluating summarization quality...


Output()

2/4 Evaluating coherence/clarity...


Output()

3/4 Evaluating tonality consistency...


Output()

4/4 Evaluating safety...



✓ Evaluation complete!

ENHANCED SUMMARY - EVALUATION RESULTS

📊 SUMMARIZATION METRIC
   Score: 1.000
   Reason: The score is 1.00 because the summary perfectly aligns with the original text, with no contradictions or extraneous information. This indicates a high-quality summarization that accurately reflects the original content.

📊 COHERENCE METRIC
   Score: 0.902
   Reason: The summary uses clear and direct language, making it easy to understand. Ideas flow logically with appropriate transitions, such as moving from self-awareness to career planning. Complex concepts like feedback analysis and alignment of values are explained clearly. There are no contradictions or confusing passages, and the structure is consistent throughout. However, a minor shortcoming is the lack of explicit transitions between some points, which slightly affects the flow.

📊 TONALITY METRIC
   Score: 0.918
   Reason: The summary maintains a consistent 'Formal Academic Writing' tone throughout, using appropri

In [25]:
import pandas as pd

# Create comparison dataframe
comparison_data = {
    'Metric': ['Summarization', 'Coherence', 'Tonality', 'Safety', 'AVERAGE'],
    'Original Score': [
        evaluation_results.SummarizationScore,
        evaluation_results.CoherenceScore,
        evaluation_results.TonalityScore,
        evaluation_results.SafetyScore,
        avg_score
    ],
    'Enhanced Score': [
        enhanced_evaluation.SummarizationScore,
        enhanced_evaluation.CoherenceScore,
        enhanced_evaluation.TonalityScore,
        enhanced_evaluation.SafetyScore,
        enhanced_avg_score
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Improvement'] = comparison_df['Enhanced Score'] - comparison_df['Original Score']
comparison_df['% Change'] = (comparison_df['Improvement'] / comparison_df['Original Score'] * 100).round(2)

print("=" * 80)
print("SCORE COMPARISON: ORIGINAL vs ENHANCED")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Determine if there was overall improvement
if enhanced_avg_score > avg_score:
    print(f"\n✅ IMPROVEMENT ACHIEVED!")
    print(f"   Overall score increased by {(enhanced_avg_score - avg_score):.3f} points")
    print(f"   ({((enhanced_avg_score - avg_score) / avg_score * 100):.2f}% improvement)")
elif enhanced_avg_score == avg_score:
    print(f"\n➡️  NO CHANGE")
    print(f"   Overall score remained the same at {avg_score:.3f}")
else:
    print(f"\n⚠️  SCORE DECREASED")
    print(f"   Overall score decreased by {(avg_score - enhanced_avg_score):.3f} points")
    print(f"   ({((avg_score - enhanced_avg_score) / avg_score * 100):.2f}% decrease)")

print("\n" + "=" * 80)

SCORE COMPARISON: ORIGINAL vs ENHANCED
       Metric  Original Score  Enhanced Score  Improvement  % Change
Summarization        0.833333        1.000000     0.166667     20.00
    Coherence        0.902033        0.902298     0.000265      0.03
     Tonality        0.914805        0.918243     0.003438      0.38
       Safety        1.000000        1.000000     0.000000      0.00
      AVERAGE        0.912543        0.955135     0.042592      4.67

✅ IMPROVEMENT ACHIEVED!
   Overall score increased by 0.043 points
   (4.67% improvement)



In [27]:
print("=" * 80)
print("DETAILED METRIC COMPARISON")
print("=" * 80)

metrics = [
    ("Summarization", evaluation_results.SummarizationScore, enhanced_evaluation.SummarizationScore),
    ("Coherence", evaluation_results.CoherenceScore, enhanced_evaluation.CoherenceScore),
    ("Tonality", evaluation_results.TonalityScore, enhanced_evaluation.TonalityScore),
    ("Safety", evaluation_results.SafetyScore, enhanced_evaluation.SafetyScore)
]

for metric_name, original_score, enhanced_score in metrics:
    improvement = enhanced_score - original_score
    
    print(f"\n{metric_name.upper()}:")
    print(f"  Original:  {original_score:.3f}")
    print(f"  Enhanced:  {enhanced_score:.3f}")
    print(f"  Change:    {improvement:+.3f}", end="")
    
    if improvement > 0:
        print(f" ✅ (Improved by {(improvement/original_score*100):.2f}%)")
    elif improvement == 0:
        print(f" ➡️  (No change)")
    else:
        print(f" ⚠️  (Decreased by {(abs(improvement)/original_score*100):.2f}%)")

print("\n" + "=" * 80)



DETAILED METRIC COMPARISON

SUMMARIZATION:
  Original:  0.833
  Enhanced:  1.000
  Change:    +0.167 ✅ (Improved by 20.00%)

COHERENCE:
  Original:  0.902
  Enhanced:  0.902
  Change:    +0.000 ✅ (Improved by 0.03%)

TONALITY:
  Original:  0.915
  Enhanced:  0.918
  Change:    +0.003 ✅ (Improved by 0.38%)

SAFETY:
  Original:  1.000
  Enhanced:  1.000
  Change:    +0.000 ➡️  (No change)



In [28]:
### Step 5: Analysis and Reflection

**Did we get a better output?**

Based on the comparison above, we can analyze whether the enhancement process successfully improved the summary quality. The results will show whether incorporating evaluation feedback into a second iteration produces measurable improvements.

**Why did we see these results?**

Several factors influence the enhancement results:

1. **Feedback Specificity**: The quality of improvement depends on how specific and actionable the evaluation feedback is. Well-defined issues in the evaluation reasons enable more targeted enhancements.

2. **Initial Quality**: If the original summary already scored highly (e.g., > 0.9 on most metrics), there may be limited room for improvement, and changes might be minor or even introduce new issues.

3. **Evaluation Consistency**: LLM-based evaluation can have some variance. Small score differences (< 0.05) may reflect evaluation uncertainty rather than meaningful quality differences.

4. **Trade-offs**: Improving one aspect (e.g., coherence) might sometimes affect another (e.g., conciseness). The enhancement process attempts to balance multiple objectives.

5. **Prompt Engineering**: The enhancement prompt explicitly includes evaluation scores and feedback, directing the model to address specific weaknesses while maintaining strengths.

**Are these controls enough?**

While this self-correction system demonstrates improvement capability, several limitations exist:

**Strengths:**
- ✅ Automated feedback loop enables iterative refinement
- ✅ Structured evaluation provides measurable quality metrics
- ✅ Multiple evaluation dimensions (summarization, coherence, tonality, safety) ensure comprehensive assessment
- ✅ Explicit feedback incorporation guides targeted improvements

**Limitations:**
- ⚠️ **Single Iteration**: Only one enhancement cycle is performed. Multiple iterations might yield better results.
- ⚠️ **Evaluation Reliability**: LLM-based evaluation can have biases and inconsistencies. Human evaluation would provide additional validation.
- ⚠️ **No Human-in-the-Loop**: Critical applications benefit from human review and approval at key stages.
- ⚠️ **Limited Context**: Very long documents may exceed context windows, affecting both summarization and evaluation quality.
- ⚠️ **Cost Considerations**: Multiple enhancement iterations increase API costs significantly.
- ⚠️ **Convergence Uncertainty**: There's no guarantee that continued iterations will improve quality; they might oscillate or plateau.

**Recommendations for Production Systems:**

1. **Multi-Stage Review**: Implement human review for high-stakes summaries
2. **Threshold-Based Enhancement**: Only trigger enhancement when scores fall below specific thresholds
3. **Iteration Limits**: Set maximum enhancement iterations to prevent excessive API costs
4. **A/B Testing**: Compare automated summaries against human-written versions
5. **Domain-Specific Metrics**: Add custom evaluation criteria relevant to your specific use case
6. **Feedback Aggregation**: Collect user feedback to continuously improve prompts and evaluation criteria
7. **Hybrid Approach**: Combine automated evaluation with periodic human audits

**Conclusion:**

This enhancement system provides a solid foundation for automated summary improvement, but should be viewed as one component of a larger quality assurance process rather than a complete solution.

SyntaxError: invalid character '✅' (U+2705) (2255141449.py, line 26)

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
